In [ ]:
import os 
import mysql.connector
from dotenv import load_dotenv

load_dotenv()

# ---------- Connection ----------

conn = mysql.connector.connect(
    host=os.getenv('DB_HOST'),
    user=os.getenv('DB_USER'),
    database=os.getenv('DB_NAME'),
    password=os.getenv('DB_PASSWORD')
)

if conn.is_connected():
    print('Connection Succesfull')

cur = conn.cursor(buffered=True) 

# ---------- Functions ----------
def view_employees():
    cur.execute('SELECT * FROM emp')
    rows = cur.fetchall()
    if len(rows) == 0:
        print('Table is empty')
    else:
        for row in rows:
            print(row)


def add_employee(emp_id, name, city, salary):
    try:
        cur.execute(
            'INSERT INTO emp (id, name, city, salary) VALUES (%s, %s, %s, %s)',
            (emp_id, name, city, salary)
        )
        conn.commit()
        print('Data added successfully ✅')
    except mysql.connector.Error as err:
        conn.rollback()
        print('Error ❌:', err)


def search_employee_by_name(name):
    cur.execute('SELECT * FROM emp WHERE name = %s', (name,))
    rows = cur.fetchall()
    if len(rows) == 0:
        print('No name found ❌')
    else:
        for row in rows:
            print(row)


def update_salary(emp_id, salary):
    try:
        cur.execute(
            'UPDATE emp SET salary = %s WHERE id = %s',
            (salary, emp_id)
        )
        conn.commit()
        if cur.rowcount == 0:
            print('ID not present ❌')
        else:
            print('Salary updated ✅')
    except mysql.connector.Error as err:
        conn.rollback()
        print('Error ❌:', err)


def delete_employee(emp_id):
    confirm = input(f'You want to delete {emp_id} (y/n)? ')
    if confirm.lower() != 'y':
        print('Deletion cancelled')
        return

    try:
        cur.execute('DELETE FROM emp WHERE id = %s', (emp_id,))
        conn.commit()
        if cur.rowcount == 0:
            print('ID not present ❌')
        else:
            print('Deletion successful ✅')
    except mysql.connector.Error as err:
        conn.rollback()
        print('Error ❌:', err)


def show_summary():
    cur.execute('SELECT COUNT(*) FROM emp')
    total = cur.fetchone()[0]

    if total == 0:
        print('Table is empty')
        return

    print('Total employees:', total)

    cur.execute('SELECT AVG(salary), MIN(salary), MAX(salary) FROM emp')
    avg_sal, min_sal, max_sal = cur.fetchone()
    print('Average salary:', round(avg_sal, 2))
    print('Min salary:', min_sal)
    print('Max salary:', max_sal)

    print('\nCity-wise:')
    cur.execute('SELECT city, COUNT(*), AVG(salary) FROM emp GROUP BY city')
    for city, count, avg in cur.fetchall():
        print(f'  {city}: {count} employees, avg salary {round(avg, 2)}')


# ---------- Menu ----------
def menu():
    while True:
        print('\n===== Employee Management System =====')
        print('1. Add employee')
        print('2. View all employees')
        print('3. Search employee')
        print('4. Update salary')
        print('5. Delete employee')
        print('6. Summary')
        print('7. Exit')

        choice = input('Choose option: ')

        try:
            if choice == '1':
                emp_id = int(input('ID: '))
                name = input('Name: ')
                city = input('City: ')
                salary = float(input('Salary: '))
                add_employee(emp_id, name, city, salary)
            elif choice == '2':
                view_employees()
            elif choice == '3':
                name = input('Name to search: ')
                search_employee_by_name(name)
            elif choice == '4':
                emp_id = int(input('ID: '))
                salary = float(input('New salary: '))
                update_salary(emp_id, salary)
            elif choice == '5':
                emp_id = int(input('ID to delete: '))
                delete_employee(emp_id)
            elif choice == '6':
                show_summary()
            elif choice == '7':
                print('Bye!')
                break
            else:
                print('Invalid option, 1 se 7 ke beech chuno')
        except ValueError:
            print('Galat input! ID aur salary number honi chahiye ❌')

    cur.close()
    conn.close()


menu()